# 03 - Market Basket Analysis
Goal: find products that are frequently bought together in UK invoices (FP-Growth), turn them into association rules (support, confidence, lift), remove trivial rules, and translate the strongest ones into cross-sell recommendations.

## 1. Load data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from mlxtend.frequent_patterns import fpgrowth, association_rules

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 60)

PROJECT_DIR = Path.cwd().parent
df = pd.read_parquet(PROJECT_DIR / "data" / "processed"/ "online_retail_clean.parquet")

uk = df[df["Country"] == "United Kingdom"]
print("UK lines:   ", f"{len(uk):,}")
print("UK invoices:", f"{uk['Invoice'].nunique():,}")
print("UK products:", f"{uk['StockCode'].nunique():,}")

UK lines:    918,363
UK invoices: 35,959
UK products: 4,706


## 2. Baskets

In [2]:
size = uk.groupby("Invoice")["StockCode"].nunique()

print(size.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1))
print("\nSingle-product baskets:", f"{(size == 1).mean():.1%}")
for limit in [100, 200, 300]:
    print(f"Baskets with > {limit} products: {(size > limit).sum():,} ({(size > limit).mean():.2%})")

count    35959.0
mean        25.2
std         43.7
min          1.0
50%         15.0
90%         51.0
95%         74.0
99%        204.0
max       1106.0
Name: StockCode, dtype: float64

Single-product baskets: 8.3%
Baskets with > 100 products: 1,136 (3.16%)
Baskets with > 200 products: 378 (1.05%)
Baskets with > 300 products: 180 (0.50%)


In [3]:
MIN_SUPPORT = 0.01
n_baskets = uk["Invoice"].nunique()

item_support = uk.groupby("StockCode")["Invoice"].nunique() / n_baskets
frequent_items = item_support[item_support >= MIN_SUPPORT].index

print("Products total:         ", len(item_support))
print(f"Products with support >= {MIN_SUPPORT:.0%}:", len(frequent_items))
print("Min baskets needed:     ", int(np.ceil(MIN_SUPPORT * n_baskets)))

Products total:          4706
Products with support >= 1%: 701
Min baskets needed:      360


In [4]:
MAX_ITEMS = 200
keep = size[size <= MAX_ITEMS].index
uk_b = uk[uk["Invoice"].isin(keep)]

print("Baskets removed:", f"{len(size) - len(keep):,} ({1 - len(keep) / len(size):.2%})")
print("Lines removed:  ", f"{1 - len(uk_b) / len(uk):.1%} of UK lines")

# recompute support on the capped baskets
n_baskets = uk_b["Invoice"].nunique()
item_support = uk_b.groupby("StockCode")["Invoice"].nunique() / n_baskets
frequent_items = item_support[item_support >= MIN_SUPPORT].index
print("Baskets:", f"{n_baskets:,}","| Frequent products:", len(frequent_items))

Baskets removed: 378 (1.05%)
Lines removed:   14.2% of UK lines
Baskets: 35,581 | Frequent products: 595


Baskets with more than 200 products (378 = 1.05% of baskets) contain 14.2% of all UK lines. Such wholesale stock-up orders link almost every product with every other and would create fake associations, so they are excluded from the basket analysis. Single-product baskets (8.3%) are kept so that support is not inflated.

## 3. Basket matrix

In [5]:
basket = (uk_b[uk_b["StockCode"].isin(frequent_items)]
          .drop_duplicates(["Invoice", "StockCode"])
          .assign(bought = True)
          .pivot(index = "Invoice", columns = "StockCode", values = "bought")
          .notna()
          .reindex(uk_b["Invoice"].unique(), fill_value = False)) # keep All baskets

print("Matrix shape:", basket.shape)
print("Top item support:", round(basket.mean().max(), 3))

Matrix shape: (35581, 595)
Top item support: 0.14


35,581 baskets x 595 products (support >= 1%). Products below 1% cannot be part of any frequent itemset (Apriori property), so removing them first gives the same result much faster. All baskets are kept as rows (reindex) so support is measured against the true number of baskets.

## 4. Frequent itemsets

In [6]:
%%time
itemsets = fpgrowth(basket, min_support=MIN_SUPPORT, use_colnames=True)
itemsets["length"] = itemsets["itemsets"].apply(len)
print(itemsets["length"].value_counts().sort_index())

length
1    595
2    327
3     85
4      1
Name: count, dtype: int64
CPU times: total: 6.34 s
Wall time: 6.36 s


FP-Growth found 595 frequent products, 327 pairs, 85 triples and 1 group of four (support >= 1%) in a few seconds. The most frequent pairs are almost all the same product in different designs (jumbo bags, lunch bags): wholesalers buy assortments.

In [7]:
names = uk_b.drop_duplicates("StockCode").set_index("StockCode")["Description"]
itemsets["items"] = itemsets["itemsets"].apply(lambda s: " + ".join(names[c] for c in s))

itemsets[itemsets["length"] == 2].nlargest(15, "support")[["support", "items"]]

,support,items
923,0.034906,JUMBO BAG RED RETROSPOT + JUMBO BAG PINK POLKADOT
755,0.031730,JUMBO BAG RED RETROSPOT + JUMBO STORAGE BAG SUKI
648,0.031337,WHITE HANGING HEART T-LIGHT HOLDER + RED HANGING HEART T...
787,0.029510,LUNCH BAG BLACK SKULL. + LUNCH BAG RED RETROSPOT
831,0.029032,WOODEN FRAME ANTIQUE WHITE + WOODEN PICTURE FRAME WHITE...
847,0.028808,JUMBO BAG STRAWBERRY + JUMBO BAG RED RETROSPOT
888,0.028189,LUNCH BAG SUKI DESIGN + LUNCH BAG RED RETROSPOT
942,0.028161,JUMBO BAG RED RETROSPOT + JUMBO SHOPPER VINTAGE RED PAISLEY
905,0.027992,LUNCH BAG RED RETROSPOT + LUNCH BAG PINK POLKADOT
819,0.027262,JUMBO BAG BAROQUE BLACK WHITE + JUMBO BAG RED RETROSPOT


## 5. Association rules

In [8]:
rules = association_rules(itemsets[["support", "itemsets"]],
                          num_itemsets=len(basket),
                          metric="lift", min_threshold=1)

rules["antecedent"] = rules["antecedents"].apply(lambda s: " + ".join(names[c] for c in s))
rules["consequent"] = rules["consequents"].apply(lambda s: " + ".join(names[c] for c in s))

print("Rules with lift > 1:", len(rules))
print(rules[["support", "confidence", "lift"]].describe().round(3))

Rules with lift > 1: 1178
        support  confidence      lift
count  1178.000    1178.000  1178.000
mean      0.013       0.375    10.406
std       0.004       0.182     7.597
min       0.010       0.072     1.305
25%       0.011       0.224     6.373
50%       0.012       0.351     9.033
75%       0.014       0.518    11.322
max       0.035       0.900    55.102


In [9]:
import re
COLOURS = {"RED", "PINK", "BLUE", "WHITE", "BLACK", "GREEN", "IVORY", "CREAM",
           "YELLOW", "ORANGE", "PURPLE", "GOLD", "SILVER", "BROWN", "GREY", "NATURAL"}
STOP = {"OF", "SET", "PACK", "THE", "AND", "WITH", "IN", "DESIGN"} | COLOURS

def words(text):
    return {w for w in re.findall(r"[A-Z]+", text.upper()) if len(w) > 1} - STOP

FAMILY_KEYS = {"JUMBO", "SIGN"}

def similar(a, b, threshold=1/3):
    wa, wb = words(a), words(b)
    if not wa or not wb:
        return False
    if wa & wb & FAMILY_KEYS:          # both share a known family keyword
        return True
    return len(wa & wb) / min(len(wa), len(wb)) >= threshold

def same_family_rule(ante, cons):
    # same family if ANY antecedent item is similar to ANY consequent item
    return any(similar(names[a], names[c]) for a in ante for c in cons)

rules["same_family"] = [same_family_rule(a, c)
                        for a, c in zip(rules["antecedents"], rules["consequents"])]

print("Same-family rules: ", rules["same_family"].sum(), f"({rules['same_family'].mean():.0%})")
print("Cross-family rules:", (~rules["same_family"]).sum())

rules.sample(20, random_state=1)[["antecedent", "consequent", "lift", "same_family"]]

Same-family rules:  1052 (89%)
Cross-family rules: 126


,antecedent,consequent,lift,same_family
660,JUMBO BAG RED RETROSPOT + JUMBO BAG PINK POLKADOT,JUMBO BAG BAROQUE BLACK WHITE,8.173098,True
548,SET/6 RED SPOTTY PAPER CUPS,SET/6 RED SPOTTY PAPER PLATES,38.416226,True
1177,GARDENERS KNEELING PAD CUP OF TEA,GARDENERS KNEELING PAD KEEP CALM,32.012823,True
321,JUMBO BAG SCANDINAVIAN PAISLEY,JUMBO BAG RED RETROSPOT + JUMBO STORAGE BAG SUKI,9.412815,True
1053,FELTCRAFT CUSHION BUTTERFLY,FELTCRAFT CUSHION OWL,25.431908,True
142,LUNCH BAG SPACEBOY DESIGN,LUNCH BAG CARS BLUE + LUNCH BAG SUKI DESIGN,10.756421,True
672,JUMBO BAG BAROQUE BLACK WHITE + JUMBO BAG RED RETROSPOT,JUMBO SHOPPER VINTAGE RED PAISLEY,8.134694,True
1132,60 CAKE CASES VINTAGE CHRISTMAS,PAPER CHAIN KIT VINTAGE CHRISTMAS,11.590389,True
861,LUNCH BAG SPACEBOY DESIGN,JUMBO BAG RED RETROSPOT,2.330949,True
547,PAINTED METAL PEARS ASSORTED,ASSORTED COLOUR BIRD ORNAMENT,10.156014,False


There is no product category column, so families are estimated from names: two products are the same family if they share at least a third of their meaningful words (colours and filler words removed) or a known family keyword (JUMBO, SIGN, added after reviewing the top rules by eye). 89% of rules are same-family. A manual check of 20 rules found ~2 wrong tags. In a bank this step is unnecessary, because product types (card, loan, deposit) are known.

In [10]:
rules["pair"] = [frozenset(a | c) for a, c in zip(rules["antecedents"], rules["consequents"])]

cross = (rules[~rules["same_family"]]
         .sort_values("confidence", ascending=False)
         .drop_duplicates("pair"))        # one directuon per product group

cols = ["antecedent", "consequent", "support", "confidence", "lift"]
top = cross[cross["confidence"] >= 0.3].nlargest(20, "lift")
print("Cross-family links (unique):", len(cross))
top[cols].round(3)

Cross-family links (unique): 63


,antecedent,consequent,support,confidence,lift
782,RED RETROSPOT SUGAR JAM BOWL,RED RETROSPOT SMALL MILK JUG,0.011,0.639,23.516
885,TOY TIDY PINK POLKADOT,RECYCLING BAG RETROSPOT,0.014,0.550,15.504
547,PAINTED METAL PEARS ASSORTED,ASSORTED COLOUR BIRD ORNAMENT,0.014,0.732,10.156
407,WOOD 2 DRAWER CABINET WHITE FINISH,WOODEN PICTURE FRAME WHITE FINISH,0.013,0.425,8.301
403,WOOD 2 DRAWER CABINET WHITE FINISH,WOODEN FRAME ANTIQUE WHITE,0.013,0.434,8.217
395,WOOD S/3 CABINET ANT WHITE FINISH,WOODEN FRAME ANTIQUE WHITE,0.013,0.403,7.622
398,WOOD S/3 CABINET ANT WHITE FINISH,WOODEN PICTURE FRAME WHITE FINISH,0.012,0.376,7.356
887,TOY TIDY PINK POLKADOT,JUMBO BAG RED RETROSPOT,0.011,0.435,4.481
682,WOODEN FRAME ANTIQUE WHITE + WOODEN PICTURE FRAME WHITE...,WHITE HANGING HEART T-LIGHT HOLDER,0.013,0.445,3.182
558,ZINC METAL HEART DECORATION,WHITE HANGING HEART T-LIGHT HOLDER,0.014,0.403,2.883


In [11]:
opp = top.copy()
opp["baskets_both"]   = (opp["support"] * n_baskets).round().astype(int)
opp["baskets_with_A"] = (opp["support"] / opp["confidence"] * n_baskets).round().astype(int)
opp["A_without_B"]    = opp["baskets_with_A"] - opp["baskets_both"]  # missed opportunity
opp[["antecedent", "consequent", "confidence", "lift",
     "baskets_with_A", "baskets_both", "A_without_B"]].round(2)

,antecedent,consequent,confidence,lift,baskets_with_A,baskets_both,A_without_B
782,RED RETROSPOT SUGAR JAM BOWL,RED RETROSPOT SMALL MILK JUG,0.64,23.52,629,402,227
885,TOY TIDY PINK POLKADOT,RECYCLING BAG RETROSPOT,0.55,15.50,914,503,411
547,PAINTED METAL PEARS ASSORTED,ASSORTED COLOUR BIRD ORNAMENT,0.73,10.16,675,494,181
407,WOOD 2 DRAWER CABINET WHITE FINISH,WOODEN PICTURE FRAME WHITE FINISH,0.42,8.30,1048,445,603
403,WOOD 2 DRAWER CABINET WHITE FINISH,WOODEN FRAME ANTIQUE WHITE,0.43,8.22,1048,455,593
395,WOOD S/3 CABINET ANT WHITE FINISH,WOODEN FRAME ANTIQUE WHITE,0.40,7.62,1172,472,700
398,WOOD S/3 CABINET ANT WHITE FINISH,WOODEN PICTURE FRAME WHITE FINISH,0.38,7.36,1172,441,731
887,TOY TIDY PINK POLKADOT,JUMBO BAG RED RETROSPOT,0.44,4.48,914,398,516
682,WOODEN FRAME ANTIQUE WHITE + WOODEN PICTURE FRAME WHITE...,WHITE HANGING HEART T-LIGHT HOLDER,0.45,3.18,1033,460,573
558,ZINC METAL HEART DECORATION,WHITE HANGING HEART T-LIGHT HOLDER,0.40,2.88,1264,510,754


Mirror rules (A->B and B->A have the same lift) are reduced to one direction: 63 unique cross-family links. The white hanging heart T-light holder ends 10 of the top 17 rules with lift of only 2-3, because it is in 14% of all baskets: popular products "follow" everything. Strong cross-sell candidates combine high lift with confidence >= 30%.

## 6. Business recommendations

These five links are the most useful for cross-selling. Most of the strongest rules in the data are the same product in a different colour (wholesalers buying assortments), so I focused on pairs of *different* products that customers clearly buy together.

**1. Red retrospot sugar bowl and milk jug.** This is the strongest link in the data: almost two out of three customers who buy sugar bowl also buy the matching milk jug, about 23 times more often than in an average basket. They are clearly buying a tea set. Still, 227 baskets had the bowl without the jug. Selling the two as a bundle, and showing the jug on the bowl's product page, would make that easier.

**2. Toy tidy and recycling bag (retrospot).** Both are home storage in the same retro print, and 55% of toy tidy buyers also take the recycling bag. With 411 baskets that had the toy tidy only, this looks more like a style collection than a strict pair. I would show them together in the catalogue as a "storage in retrospot" group rather than force a bundle.

**3. Painted metal pears and bird ornament.** This has the highest confidence of all the rules: 73% of baskets with the metal pears also include the bird ornament. When the link is this reliable, a simple "frequently bought together" suggestion on the pears' page is probable enough. Only 181 baskets missed it, so the gain is small but almost free to get.

**4. White wooden cabinet and white picture frame.** Customers furnishing a room in the "white finish" style buy these together (lift 8.3). But 603 baskets had the cabinet without the frame, which is a fairly large gap. Placing the white furniture and frames next to each other in the catalogue, as one room look, should help customers find the rest of the set.

**5 White picture frame and white hanging heart T-light holder.** The link itself is weaker (lift 2.9), because the T-light holder is already in 14% of all baskets. But it has the biggest reach: 1,089 baskets had the frame without the T-light holder. That is a good case for a low-cost email to frame buyers rather than a bundle. It is not the strongest rule, but it reaches the most customers.

**A note of caution.** "Baskets without the second product" is the number of customers we could target, not guaranteed extra sales. Even for the best rule, a third of bowl buyers did not want the jug. The real effect of any of these actions should be tested first, for example by sending an offer to half of the target customers and comparing their purchases with the other half.

## 7. Save

In [12]:
out = rules[["antecedents", "consequents", "antecedent", "consequent",
             "support", "confidence", "lift", "same_family"]].copy()
out["antecedents"] = out["antecedents"].apply(lambda s: "|".join(sorted(s))) # frozenset -> text
out["consequents"] = out["consequents"].apply(lambda s: "|".join(sorted(s)))

OUT = PROJECT_DIR / "data" / "processed" / "basket_rules.parquet"
out.to_parquet(OUT, index=False)
print("Saved:", OUT.name, out.shape)

Saved: basket_rules.parquet (1178, 8)
